In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

print("PROJECT DIRECTORY:", PROJECT_DIR.resolve())
print("DATA DIRECTORY:", DATA_DIR.resolve())
print("DATA DIRECTORY EXISTS:", DATA_DIR.exists())

if DATA_DIR.exists():
    print("\nAVAILABLE CSV FILES:")
    for path in sorted(DATA_DIR.glob("*.csv")):
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{path.name:20s} {size_mb:8.2f} MB")
else:
    print("\nData folder not found. Check the notebook's working directory.")

PROJECT DIRECTORY: C:\Users\Administrator\Self Study\NexusIQ
DATA DIRECTORY: C:\Users\Administrator\Self Study\NexusIQ\data
DATA DIRECTORY EXISTS: True

AVAILABLE CSV FILES:
champs.csv               0.00 MB
matches.csv             11.27 MB
participants.csv        90.47 MB
stats1.csv             298.63 MB
stats2.csv             249.46 MB
teambans.csv            25.20 MB
teamstats.csv           20.60 MB


In [3]:
import pandas as pd

# Match metadata
matches = pd.read_csv(
    DATA_DIR / "matches.csv",
    usecols=[
        "id", "gameid", "platformid", "queueid",
        "seasonid", "duration", "creation", "version"
    ]
)

# Participant and champion-select information
participants = pd.read_csv(
    DATA_DIR / "participants.csv",
    usecols=[
        "id", "matchid", "player", "championid",
        "ss1", "ss2", "role", "position"
    ]
)

champions = pd.read_csv(DATA_DIR / "champs.csv")

print("MATCHES:", matches.shape)
print("PARTICIPANTS:", participants.shape)
print("CHAMPIONS:", champions.shape)

print("\nMATCH SAMPLE")
display(matches.head())

print("\nPARTICIPANT SAMPLE")
display(participants.head())

print("\nCHAMPION SAMPLE")
display(champions.head())

MATCHES: (184069, 8)
PARTICIPANTS: (1834520, 8)
CHAMPIONS: (138, 2)

MATCH SAMPLE


,id,gameid,platformid,queueid,seasonid,duration,creation,version
0,10,3187427022,EUW1,420,8,1909,1495068946860,7.10.187.9675
1,11,3187425281,EUW1,420,8,1693,1495066760778,7.10.187.9675
2,12,3187269801,EUW1,420,8,1482,1495053375889,7.10.187.9675
3,13,3187252065,EUW1,420,8,1954,1495050993613,7.10.187.9675
4,14,3187201038,EUW1,420,8,2067,1495047893400,7.10.187.9675



PARTICIPANT SAMPLE


,id,matchid,player,championid,ss1,ss2,role,position
0,9,10,1,19,4,11,NONE,JUNGLE
1,10,10,2,267,3,4,DUO_SUPPORT,BOT
2,11,10,3,119,7,4,DUO_CARRY,BOT
3,12,10,4,114,12,4,SOLO,TOP
4,13,10,5,112,4,3,SOLO,MID



CHAMPION SAMPLE


,name,id
0,Jax,24
1,Sona,37
2,Tristana,18
3,Varus,110
4,Fiora,114


In [4]:
# Select ranked matches with exactly 10 participant records
ranked_match_ids = set(
    matches.loc[matches["queueid"] == 420, "id"]
)

ranked_participants = participants[
    participants["matchid"].isin(ranked_match_ids)
].copy()

participant_counts = ranked_participants.groupby("matchid").agg(
    participant_rows=("id", "size"),
    unique_players=("player", "nunique"),
    unique_champions=("championid", "nunique"),
    missing_champions=("championid", lambda s: s.isna().sum()),
)

valid_match_ids = participant_counts.index[
    (participant_counts["participant_rows"] == 10)
    & (participant_counts["unique_players"] == 10)
    & (participant_counts["unique_champions"] == 10)
    & (participant_counts["missing_champions"] == 0)
]

print("RANKED MATCHES:", len(ranked_match_ids))
print("RANKED MATCHES WITH PARTICIPANT DATA:", ranked_participants["matchid"].nunique())
print("MATCHES WITH 10 UNIQUE PLAYERS AND CHAMPIONS:", len(valid_match_ids))

print("\nPARTICIPANT FIELDS AVAILABLE BEFORE THE MATCH")
display(
    ranked_participants[
        ["championid", "ss1", "ss2", "role", "position"]
    ].isna().sum().to_frame("missing_values")
)

print("\nROLE VALUES")
display(ranked_participants["role"].value_counts(dropna=False).to_frame("rows"))

print("\nPOSITION VALUES")
display(ranked_participants["position"].value_counts(dropna=False).to_frame("rows"))

RANKED MATCHES: 161078
RANKED MATCHES WITH PARTICIPANT DATA: 161078
MATCHES WITH 10 UNIQUE PLAYERS AND CHAMPIONS: 161075

PARTICIPANT FIELDS AVAILABLE BEFORE THE MATCH


,missing_values
championid,0
ss1,0
ss2,0
role,0
position,0



ROLE VALUES


,rows
role,
SOLO,632313
NONE,328650
DUO_SUPPORT,298179
DUO_CARRY,298178
DUO,53442



POSITION VALUES


,rows
position,
BOT,640055
JUNGLE,325407
MID,324969
TOP,320331


In [5]:
# Load only the participant ID and final win label.
# These files are large, so avoid loading their other columns.
stats1_target = pd.read_csv(
    DATA_DIR / "stats1.csv",
    usecols=["id", "win"]
)

stats2_target = pd.read_csv(
    DATA_DIR / "stats2.csv",
    usecols=["id", "win"]
)

stats_target = pd.concat(
    [stats1_target, stats2_target],
    ignore_index=True
)

del stats1_target, stats2_target

# Keep only ranked matches with 10 unique players and champions
valid_participants = ranked_participants[
    ranked_participants["matchid"].isin(valid_match_ids)
].copy()

labeled_participants = valid_participants.merge(
    stats_target,
    on="id",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("VALID PARTICIPANT ROWS:", len(valid_participants))
print("\nLABEL JOIN STATUS")
display(labeled_participants["_merge"].value_counts().to_frame("rows"))

print("\nWIN LABEL COUNTS")
display(labeled_participants["win"].value_counts(dropna=False).to_frame("rows"))

match_labels = labeled_participants.groupby("matchid").agg(
    participant_rows=("id", "size"),
    labeled_rows=("win", "count"),
    distinct_win_labels=("win", "nunique"),
    total_wins=("win", "sum"),
)

print("\nMATCH LABEL QUALITY")
display(match_labels["labeled_rows"].value_counts().sort_index().to_frame("matches"))

print("\nMATCHES WITH EXACTLY FIVE WINS AND FIVE LOSSES:",
      ((match_labels["labeled_rows"] == 10) &
       (match_labels["total_wins"] == 5) &
       (match_labels["distinct_win_labels"] == 2)).sum())

VALID PARTICIPANT ROWS: 1610750

LABEL JOIN STATUS


,rows
_merge,
both,1610749
left_only,1
right_only,0



WIN LABEL COUNTS


,rows
win,
1.0,805375
0.0,805374
NaN,1



MATCH LABEL QUALITY


,matches
labeled_rows,
9,1
10,161074



MATCHES WITH EXACTLY FIVE WINS AND FIVE LOSSES: 161074


In [6]:
# Keep only matches with 10 participant rows, all labels present,
# and exactly five wins and five losses.
clean_match_ids = match_labels.index[
    (match_labels["participant_rows"] == 10)
    & (match_labels["labeled_rows"] == 10)
    & (match_labels["total_wins"] == 5)
    & (match_labels["distinct_win_labels"] == 2)
]

clean_players = labeled_participants[
    labeled_participants["matchid"].isin(clean_match_ids)
].copy()

# Player slots 1-5 are provisionally treated as Blue Side;
# slots 6-10 as Red Side. Verify this convention before modeling.
clean_players["side"] = clean_players["player"].map(
    lambda p: "blue" if p <= 5 else "red"
)

# One row per match, with champion IDs and summoner spells by slot.
draft_features = clean_players.pivot(
    index="matchid",
    columns="player",
    values=["championid", "ss1", "ss2"]
)

draft_features.columns = [
    f"{feature}_p{player}"
    for feature, player in draft_features.columns
]
draft_features = draft_features.reset_index()

# Define the target as whether the Blue Side won.
blue_wins = (
    clean_players[clean_players["side"] == "blue"]
    .groupby("matchid")["win"]
    .mean()
    .rename("blue_win")
)

match_level = (
    matches[matches["id"].isin(clean_match_ids)]
    .rename(columns={"id": "matchid"})
    .merge(draft_features, on="matchid", validate="one_to_one")
    .merge(blue_wins, on="matchid", validate="one_to_one")
)

# Keep match metadata for filtering/splitting; do not use duration as a feature.
print("MATCH-LEVEL DATASET SHAPE:", match_level.shape)
print("\nTARGET DISTRIBUTION")
display(match_level["blue_win"].value_counts(dropna=False).to_frame("matches"))

print("\nMISSING VALUES")
display(
    match_level.isna().sum()
    .loc[lambda s: s > 0]
    .to_frame("missing_values")
)

print("\nSAMPLE")
display(match_level.head())

MATCH-LEVEL DATASET SHAPE: (161074, 39)

TARGET DISTRIBUTION


,matches
blue_win,
1.0,82894
0.0,78180



MISSING VALUES


,missing_values



SAMPLE


,matchid,gameid,platformid,queueid,seasonid,duration,creation,version,championid_p1,championid_p2,...,ss2_p2,ss2_p3,ss2_p4,ss2_p5,ss2_p6,ss2_p7,ss2_p8,ss2_p9,ss2_p10,blue_win
0,10,3187427022,EUW1,420,8,1909,1495068946860,7.10.187.9675,19,267,...,4,4,4,3,4,12,4,4,4,0.0
1,11,3187425281,EUW1,420,8,1693,1495066760778,7.10.187.9675,115,69,...,7,4,12,11,4,4,1,4,11,0.0
2,12,3187269801,EUW1,420,8,1482,1495053375889,7.10.187.9675,119,40,...,4,21,11,4,4,4,12,4,7,0.0
3,13,3187252065,EUW1,420,8,1954,1495050993613,7.10.187.9675,7,222,...,7,11,4,14,7,12,11,3,14,0.0
4,14,3187201038,EUW1,420,8,2067,1495047893400,7.10.187.9675,61,6,...,4,11,7,4,7,11,3,12,14,0.0


In [7]:
# Confirm target is binary and complete
print("TARGET COUNTS:")
display(match_level["blue_win"].value_counts(dropna=False).sort_index())

print("\nCREATION TIMESTAMP RANGE:")
print("Minimum:", match_level["creation"].min())
print("Maximum:", match_level["creation"].max())

# Convert Unix milliseconds to UTC datetime for readable inspection
match_level["creation_dt"] = pd.to_datetime(
    match_level["creation"],
    unit="ms",
    utc=True,
    errors="coerce"
)

print("\nCREATION DATE RANGE:")
print("Start:", match_level["creation_dt"].min())
print("End:  ", match_level["creation_dt"].max())
print("Invalid timestamps:", match_level["creation_dt"].isna().sum())

# Chronological 80/20 split, with all rows at the same timestamp
# kept together so identical timestamps cannot cross the boundary.
ordered = match_level.sort_values(
    ["creation", "matchid"]
).reset_index(drop=True)

split_index = int(len(ordered) * 0.80)
cutoff_timestamp = ordered.loc[split_index, "creation"]

train_matches = ordered[ordered["creation"] < cutoff_timestamp].copy()
test_matches = ordered[ordered["creation"] >= cutoff_timestamp].copy()

print("\nPROVISIONAL CHRONOLOGICAL SPLIT")
print("Cutoff UTC:", pd.to_datetime(cutoff_timestamp, unit="ms", utc=True))
print("Train matches:", len(train_matches))
print("Test matches:", len(test_matches))

print("\nTARGET RATE BY SPLIT")
display(
    pd.DataFrame({
        "train_blue_win_rate": [train_matches["blue_win"].mean()],
        "test_blue_win_rate": [test_matches["blue_win"].mean()],
    }).round(4)
)

print("\nSPLIT DATE RANGES")
print("Train:", train_matches["creation_dt"].min(), "to", train_matches["creation_dt"].max())
print("Test: ", test_matches["creation_dt"].min(), "to", test_matches["creation_dt"].max())

TARGET COUNTS:


blue_win
0.0    78180
1.0    82894
Name: count, dtype: int64


CREATION TIMESTAMP RANGE:
Minimum: 1479754066334
Maximum: 1496908941036

CREATION DATE RANGE:
Start: 2016-11-21 18:47:46.334000+00:00
End:   2017-06-08 08:02:21.036000+00:00
Invalid timestamps: 0

PROVISIONAL CHRONOLOGICAL SPLIT
Cutoff UTC: 2017-05-18 16:53:51.761000+00:00
Train matches: 128859
Test matches: 32215

TARGET RATE BY SPLIT


,train_blue_win_rate,test_blue_win_rate
0,0.514,0.5172



SPLIT DATE RANGES
Train: 2016-11-21 18:47:46.334000+00:00 to 2017-05-18 16:53:51.073000+00:00
Test:  2017-05-18 16:53:51.761000+00:00 to 2017-06-08 08:02:21.036000+00:00


In [8]:
# INSPECTION ONLY — no preprocessing decisions or model training

import pandas as pd

print("1. MATCH IDENTIFIERS")
print("Duplicate match IDs:", matches["id"].duplicated().sum())
print("Duplicate game IDs:", matches["gameid"].duplicated().sum())
print("Duplicate participant IDs:", participants["id"].duplicated().sum())

print("\n2. PATCH DISTRIBUTION IN THE CLEAN MATCH SET")
patch_summary = (
    match_level.groupby("version", dropna=False)
    .agg(
        matches=("matchid", "size"),
        first_timestamp=("creation", "min"),
        last_timestamp=("creation", "max"),
        blue_win_rate=("blue_win", "mean"),
    )
    .sort_values("first_timestamp")
)

patch_summary["first_date_utc"] = pd.to_datetime(
    patch_summary["first_timestamp"], unit="ms", utc=True
)
patch_summary["last_date_utc"] = pd.to_datetime(
    patch_summary["last_timestamp"], unit="ms", utc=True
)
patch_summary["blue_win_rate_pct"] = patch_summary["blue_win_rate"] * 100

display(
    patch_summary[
        ["matches", "first_date_utc", "last_date_utc", "blue_win_rate_pct"]
    ].round({"blue_win_rate_pct": 2})
)

print("\n3. PATCH CONCENTRATION")
patch_counts = match_level["version"].value_counts(dropna=False)
print("Distinct patch versions:", match_level["version"].nunique())
print("Largest patch share:", round(patch_counts.iloc[0] / len(match_level) * 100, 2), "%")

print("\n4. CHAMPION ID VALIDITY")
known_champion_ids = set(champions["id"].dropna())
observed_champion_ids = set(
    participants.loc[
        participants["matchid"].isin(clean_match_ids), "championid"
    ].dropna()
)

print("Distinct observed champion IDs:", len(observed_champion_ids))
print("Observed IDs absent from champs.csv:",
      sorted(observed_champion_ids - known_champion_ids))
print("Champion IDs in champs.csv never observed:",
      len(known_champion_ids - observed_champion_ids))

print("\n5. PLAYER SLOT COUNTS")
display(
    participants.loc[
        participants["matchid"].isin(clean_match_ids), "player"
    ].value_counts(dropna=False).sort_index().to_frame("participant_rows")
)

print("\n6. ROLE/POSITION LABEL COMBINATIONS")
role_position_counts = (
    participants.loc[
        participants["matchid"].isin(clean_match_ids),
        ["role", "position"]
    ]
    .value_counts(dropna=False)
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(role_position_counts)

1. MATCH IDENTIFIERS
Duplicate match IDs: 0
Duplicate game IDs: 0
Duplicate participant IDs: 0

2. PATCH DISTRIBUTION IN THE CLEAN MATCH SET


,matches,first_date_utc,last_date_utc,blue_win_rate_pct
version,,,,
6.22.165.9605,5,2016-11-21 18:47:46.334000+00:00,2016-11-22 05:47:33.463000+00:00,20.00
6.23.166.2676,71,2016-11-22 17:14:06.328000+00:00,2016-12-07 05:12:45.300000+00:00,52.11
6.24.168.1268,99,2016-12-07 13:01:45.230000+00:00,2016-12-13 14:30:32.698000+00:00,39.39
6.24.169.4424,348,2016-12-13 20:47:28.851000+00:00,2017-01-11 09:16:16.747000+00:00,47.70
7.1.171.6847,79,2017-01-11 17:07:21.461000+00:00,2017-01-17 18:46:11.377000+00:00,54.43
7.1.172.6393,65,2017-01-17 20:47:07.128000+00:00,2017-01-25 08:46:15.547000+00:00,46.15
7.2.173.3544,3,2017-01-26 02:23:04.293000+00:00,2017-01-26 18:51:58.992000+00:00,66.67
7.2.174.829,131,2017-01-26 20:52:09.012000+00:00,2017-02-08 02:58:38.089000+00:00,53.44
7.3.175.271,148,2017-02-08 21:02:38.252000+00:00,2017-02-23 08:55:35.743000+00:00,50.68



3. PATCH CONCENTRATION
Distinct patch versions: 21
Largest patch share: 40.68 %

4. CHAMPION ID VALIDITY
Distinct observed champion IDs: 136
Observed IDs absent from champs.csv: []
Champion IDs in champs.csv never observed: 2

5. PLAYER SLOT COUNTS


,participant_rows
player,
1,161074
2,161074
3,161074
4,161074
5,161074
6,161074
7,161074
8,161074
9,161074



6. ROLE/POSITION LABEL COMBINATIONS


,role,position,rows
0,NONE,JUNGLE,325402
1,SOLO,TOP,318539
2,SOLO,MID,309393
3,DUO_CARRY,BOT,294343
4,DUO_SUPPORT,BOT,294343
5,DUO,BOT,44592
6,DUO,MID,7696
7,SOLO,BOT,4373
8,DUO_CARRY,MID,3538
9,DUO_SUPPORT,MID,3538


In [9]:

# INSPECTION ONLY — compare fixed-slot patterns with team-composition patterns

import pandas as pd

print("1. CHAMPION DIVERSITY BY PARTICIPANT SLOT")

slot_summary = (
    clean_players.groupby("player")
    .agg(
        unique_champions=("championid", "nunique"),
        most_common_champion_share=(
            "championid",
            lambda s: s.value_counts(normalize=True).iloc[0] * 100
        ),
        rows=("championid", "size")
    )
)

display(slot_summary.round(2))

print("\n2. ROLE DISTRIBUTION BY SLOT")

display(pd.crosstab(
    clean_players["player"],
    clean_players["role"],
    normalize="index"
).mul(100).round(1))

print("\n3. POSITION DISTRIBUTION BY SLOT")

display(pd.crosstab(
    clean_players["player"],
    clean_players["position"],
    normalize="index"
).mul(100).round(1))

print("\n4. HOW OFTEN DOES EACH CHAMPION APPEAR IN DIFFERENT SLOTS?")

champion_slot_counts = (
    clean_players.groupby("championid")["player"]
    .nunique()
)

print("Champions appearing in only one slot:",
      int((champion_slot_counts == 1).sum()))
print("Champions appearing in multiple slots:",
      int((champion_slot_counts > 1).sum()))
print("Maximum distinct slots used by one champion:",
      int(champion_slot_counts.max()))

print("\n5. TEAM COMPOSITION CONSISTENCY")

# Count distinct champions per team-side group in each match.
# This is a structural check, not an assumption about side advantage.
composition_check = clean_players.copy()
composition_check["slot_group"] = (
    composition_check["player"].map(
        lambda p: "slots_1_to_5" if p <= 5 else "slots_6_to_10"
    )
)

team_composition = (
    composition_check.groupby(["matchid", "slot_group"])
    .agg(
        picks=("championid", "size"),
        unique_picks=("championid", "nunique")
    )
)

display(team_composition.describe().round(2))


1. CHAMPION DIVERSITY BY PARTICIPANT SLOT


,unique_champions,most_common_champion_share,rows
player,,,
1,136,3.34,161074
2,136,3.37,161074
3,136,3.34,161074
4,136,3.33,161074
5,136,3.36,161074
6,136,3.30,161074
7,136,3.32,161074
8,136,3.34,161074
9,136,3.28,161074



2. ROLE DISTRIBUTION BY SLOT


role,DUO,DUO_CARRY,DUO_SUPPORT,NONE,SOLO
player,,,,,
1,3.3,17.8,18.0,20.4,40.5
2,3.4,19.3,19.0,20.4,37.9
3,3.3,19.0,18.8,20.2,38.7
4,3.4,19.2,19.1,20.3,38.1
5,3.2,17.6,17.9,20.4,40.8
6,3.3,17.4,18.0,20.4,40.8
7,3.4,19.2,18.8,20.4,38.1
8,3.4,19.1,18.7,20.3,38.6
9,3.4,19.1,18.8,20.6,38.2



3. POSITION DISTRIBUTION BY SLOT


position,BOT,JUNGLE,MID,TOP
player,,,,
1,38.3,20.2,21.0,20.4
2,41.0,20.2,19.5,19.3
3,40.3,20.0,20.0,19.6
4,40.9,20.1,19.4,19.6
5,37.9,20.2,21.1,20.7
6,38.2,20.2,21.0,20.5
7,41.1,20.1,19.5,19.3
8,40.7,20.0,19.7,19.6
9,40.8,20.3,19.6,19.3



4. HOW OFTEN DOES EACH CHAMPION APPEAR IN DIFFERENT SLOTS?
Champions appearing in only one slot: 0
Champions appearing in multiple slots: 136
Maximum distinct slots used by one champion: 10

5. TEAM COMPOSITION CONSISTENCY


,picks,unique_picks
count,322148.0,322148.0
mean,5.0,5.0
std,0.0,0.0
min,5.0,5.0
25%,5.0,5.0
50%,5.0,5.0
75%,5.0,5.0
max,5.0,5.0


In [10]:

# INSPECTION ONLY — examine patch sizes and candidate chronological boundaries

patch_timeline = (
    match_level.groupby("version", dropna=False)
    .agg(
        matches=("matchid", "size"),
        first_creation=("creation", "min"),
        last_creation=("creation", "max"),
    )
    .sort_values("first_creation")
    .reset_index()
)

patch_timeline["cumulative_matches"] = patch_timeline["matches"].cumsum()
patch_timeline["cumulative_share_pct"] = (
    patch_timeline["cumulative_matches"] / len(match_level) * 100
)

print("PATCH TIMELINE AND CUMULATIVE DATA SHARE")
display(
    patch_timeline[
        [
            "version",
            "matches",
            "cumulative_matches",
            "cumulative_share_pct",
            "first_creation",
            "last_creation",
        ]
    ].assign(
        first_date=lambda d: pd.to_datetime(
            d["first_creation"], unit="ms", utc=True
        ).dt.strftime("%Y-%m-%d"),
        last_date=lambda d: pd.to_datetime(
            d["last_creation"], unit="ms", utc=True
        ).dt.strftime("%Y-%m-%d"),
    )[
        [
            "version",
            "matches",
            "cumulative_matches",
            "cumulative_share_pct",
            "first_date",
            "last_date",
        ]
    ].round({"cumulative_share_pct": 2})
)

print("\nCANDIDATE PATCH-BOUNDARY SPLITS")
for target_share in [0.70, 0.75, 0.80, 0.85, 0.90]:
    candidates = patch_timeline[
        patch_timeline["cumulative_share_pct"] >= target_share * 100
    ]
    if candidates.empty:
        continue

    boundary = candidates.iloc[0]
    train_rows = int(boundary["cumulative_matches"])
    test_rows = len(match_level) - train_rows

    print(
        f"Target train share {target_share:.0%}: "
        f"train through {boundary['version']} "
        f"({train_rows:,} matches, {train_rows / len(match_level):.1%}); "
        f"later patches test set = {test_rows:,} matches "
        f"({test_rows / len(match_level):.1%})"
    )


PATCH TIMELINE AND CUMULATIVE DATA SHARE


,version,matches,cumulative_matches,cumulative_share_pct,first_date,last_date
0,6.22.165.9605,5,5,0.00,2016-11-21,2016-11-22
1,6.23.166.2676,71,76,0.05,2016-11-22,2016-12-07
2,6.24.168.1268,99,175,0.11,2016-12-07,2016-12-13
3,6.24.169.4424,348,523,0.32,2016-12-13,2017-01-11
4,7.1.171.6847,79,602,0.37,2017-01-11,2017-01-17
5,7.1.172.6393,65,667,0.41,2017-01-17,2017-01-25
6,7.2.173.3544,3,670,0.42,2017-01-26,2017-01-26
7,7.2.174.829,131,801,0.50,2017-01-26,2017-02-08
8,7.3.175.271,148,949,0.59,2017-02-08,2017-02-23
9,7.4.176.9828,91,1040,0.65,2017-02-23,2017-03-02



CANDIDATE PATCH-BOUNDARY SPLITS
Target train share 70%: train through 7.9.186.8155 (118,675 matches, 73.7%); later patches test set = 42,399 matches (26.3%)
Target train share 75%: train through 7.10.187.9675 (161,054 matches, 100.0%); later patches test set = 20 matches (0.0%)
Target train share 80%: train through 7.10.187.9675 (161,054 matches, 100.0%); later patches test set = 20 matches (0.0%)
Target train share 85%: train through 7.10.187.9675 (161,054 matches, 100.0%); later patches test set = 20 matches (0.0%)
Target train share 90%: train through 7.10.187.9675 (161,054 matches, 100.0%); later patches test set = 20 matches (0.0%)


In [11]:

# INSPECTION ONLY — champion coverage under candidate chronological splits

ordered_matches = match_level.sort_values(
    ["creation", "matchid"]
).reset_index(drop=True)

# Option A: train through patch 7.9.186.8155; test on later patches
option_a_train = ordered_matches[
    ordered_matches["version"] != "7.10.187.9675"
]
# Use timestamps for the actual boundary, not the filtered frame above.
option_a_cutoff = ordered_matches.loc[
    ordered_matches["version"] == "7.10.187.9675", "creation"
].min()

option_a_train = ordered_matches[
    ordered_matches["creation"] < option_a_cutoff
]
option_a_test = ordered_matches[
    ordered_matches["creation"] >= option_a_cutoff
]

# Option B: chronological 80/20 split, keeping equal timestamps together
split_index = int(len(ordered_matches) * 0.80)
option_b_cutoff = ordered_matches.loc[split_index, "creation"]

option_b_train = ordered_matches[
    ordered_matches["creation"] < option_b_cutoff
]
option_b_test = ordered_matches[
    ordered_matches["creation"] >= option_b_cutoff
]

def champion_coverage(train_df, test_df, label):
    champion_cols = [
        f"championid_p{i}" for i in range(1, 11)
    ]

    train_champions = set(train_df[champion_cols].to_numpy().ravel())
    test_champions = set(test_df[champion_cols].to_numpy().ravel())

    unseen = test_champions - train_champions

    print(f"\n{label}")
    print("Train matches:", len(train_df))
    print("Test matches:", len(test_df))
    print("Distinct champions in train:", len(train_champions))
    print("Distinct champions in test:", len(test_champions))
    print("Test champions never seen in train:", sorted(unseen))

champion_coverage(option_a_train, option_a_test, "OPTION A: PATCH-ALIGNED")
champion_coverage(option_b_train, option_b_test, "OPTION B: CHRONOLOGICAL 80/20")

print("\nTEST SET PATCH MIX")
display(
    pd.DataFrame({
        "Option A": option_a_test["version"].value_counts(),
        "Option B": option_b_test["version"].value_counts(),
    }).fillna(0).astype(int)
)



OPTION A: PATCH-ALIGNED
Train matches: 118675
Test matches: 42399
Distinct champions in train: 136
Distinct champions in test: 136
Test champions never seen in train: []

OPTION B: CHRONOLOGICAL 80/20
Train matches: 128859
Test matches: 32215
Distinct champions in train: 136
Distinct champions in test: 136
Test champions never seen in train: []

TEST SET PATCH MIX


,Option A,Option B
version,,
7.10.187.9675,42379,32195
7.11.189.942,20,20


In [12]:

# INSPECTION ONLY — summoner spell coverage and diversity

spell_cols = [
    f"{spell}_p{i}"
    for i in range(1, 11)
    for spell in ["ss1", "ss2"]
]

print("1. MISSING SUMMONER SPELL VALUES")
display(
    match_level[spell_cols]
    .isna()
    .sum()
    .to_frame("missing_values")
    .query("missing_values > 0")
)

print("Total missing spell values:", int(match_level[spell_cols].isna().sum().sum()))

print("\n2. UNIQUE SPELL IDS")
for col in spell_cols:
    print(f"{col}: {match_level[col].nunique()} distinct values")

print("\n3. MOST COMMON SPELL IDS ACROSS ALL PLAYER SLOTS")
all_spell_values = pd.concat(
    [clean_players["ss1"], clean_players["ss2"]],
    ignore_index=True
)

display(
    all_spell_values.value_counts(dropna=False)
    .rename_axis("spell_id")
    .reset_index(name="uses")
    .assign(
        share_pct=lambda d: d["uses"] / d["uses"].sum() * 100
    )
    .head(20)
    .round({"share_pct": 2})
)

print("\n4. NUMBER OF DISTINCT SPELL COMBINATIONS PER CHAMPION")
champion_spell_patterns = (
    clean_players.groupby("championid")
    .apply(
        lambda g: g[["ss1", "ss2"]].drop_duplicates().shape[0],
        include_groups=False
    )
    .rename("distinct_spell_pairs")
)

display(champion_spell_patterns.describe().round(2).to_frame())

print("\n5. MATCHES WHERE BOTH TEAMS HAVE IDENTICAL SPELL-PAIR COUNTS")
# Diagnostic only: count each team's unique spell pairs.
spell_players = clean_players.copy()
spell_players["slot_group"] = spell_players["player"].map(
    lambda p: "slots_1_to_5" if p <= 5 else "slots_6_to_10"
)
spell_players["spell_pair"] = (
    spell_players["ss1"].astype(str) + "_" +
    spell_players["ss2"].astype(str)
)

spell_pair_counts = (
    spell_players.groupby(["matchid", "slot_group"])["spell_pair"]
    .nunique()
    .unstack()
)

display(spell_pair_counts.describe().round(2))


1. MISSING SUMMONER SPELL VALUES


,missing_values


Total missing spell values: 0

2. UNIQUE SPELL IDS
ss1_p1: 9 distinct values
ss2_p1: 9 distinct values
ss1_p2: 9 distinct values
ss2_p2: 9 distinct values
ss1_p3: 9 distinct values
ss2_p3: 9 distinct values
ss1_p4: 9 distinct values
ss2_p4: 9 distinct values
ss1_p5: 9 distinct values
ss2_p5: 9 distinct values
ss1_p6: 9 distinct values
ss2_p6: 9 distinct values
ss1_p7: 9 distinct values
ss2_p7: 9 distinct values
ss1_p8: 9 distinct values
ss2_p8: 9 distinct values
ss1_p9: 9 distinct values
ss2_p9: 9 distinct values
ss1_p10: 9 distinct values
ss2_p10: 9 distinct values

3. MOST COMMON SPELL IDS ACROSS ALL PLAYER SLOTS


,spell_id,uses,share_pct
0,4,1566282,48.62
1,7,331303,10.28
2,11,322439,10.01
3,12,321333,9.97
4,14,300711,9.33
5,3,276162,8.57
6,6,52869,1.64
7,21,42582,1.32
8,1,7799,0.24



4. NUMBER OF DISTINCT SPELL COMBINATIONS PER CHAMPION


,distinct_spell_pairs
count,136.00
mean,27.76
std,6.51
min,14.00
25%,23.00
50%,27.00
75%,31.00
max,52.00



5. MATCHES WHERE BOTH TEAMS HAVE IDENTICAL SPELL-PAIR COUNTS


slot_group,slots_1_to_5,slots_6_to_10
count,161074.00,161074.00
mean,4.78,4.78
std,0.43,0.43
min,1.00,2.00
25%,5.00,5.00
50%,5.00,5.00
75%,5.00,5.00
max,5.00,5.00


In [13]:

# INSPECTION ONLY — outcome distribution by patch and candidate split

patch_outcomes = (
    match_level.groupby("version")
    .agg(
        matches=("matchid", "size"),
        blue_wins=("blue_win", "sum"),
        blue_win_rate=("blue_win", "mean"),
        first_creation=("creation", "min"),
        last_creation=("creation", "max"),
    )
    .sort_values("first_creation")
)

patch_outcomes["blue_losses"] = (
    patch_outcomes["matches"] - patch_outcomes["blue_wins"]
)
patch_outcomes["blue_win_rate_pct"] = patch_outcomes["blue_win_rate"] * 100

display(
    patch_outcomes[
        [
            "matches",
            "blue_wins",
            "blue_losses",
            "blue_win_rate_pct",
        ]
    ].round({"blue_win_rate_pct": 2})
)

print("\nOVERALL TARGET DISTRIBUTION")
display(
    match_level["blue_win"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Blue loss", 1: "Blue win"})
    .to_frame("matches")
    .assign(
        share_pct=lambda d: d["matches"] / len(match_level) * 100
    )
    .round({"share_pct": 2})
)

print("\nTARGET RATE IN CANDIDATE SPLITS")
for name, train_df, test_df in [
    ("Option A — patch-aligned", option_a_train, option_a_test),
    ("Option B — chronological 80/20", option_b_train, option_b_test),
]:
    print(f"\n{name}")
    print(f"Train Blue win rate: {train_df['blue_win'].mean():.3%}")
    print(f"Test Blue win rate:  {test_df['blue_win'].mean():.3%}")


,matches,blue_wins,blue_losses,blue_win_rate_pct
version,,,,
6.22.165.9605,5,1.0,4.0,20.00
6.23.166.2676,71,37.0,34.0,52.11
6.24.168.1268,99,39.0,60.0,39.39
6.24.169.4424,348,166.0,182.0,47.70
7.1.171.6847,79,43.0,36.0,54.43
7.1.172.6393,65,30.0,35.0,46.15
7.2.173.3544,3,2.0,1.0,66.67
7.2.174.829,131,70.0,61.0,53.44
7.3.175.271,148,75.0,73.0,50.68



OVERALL TARGET DISTRIBUTION


,matches,share_pct
blue_win,,
Blue loss,78180,48.54
Blue win,82894,51.46



TARGET RATE IN CANDIDATE SPLITS

Option A — patch-aligned
Train Blue win rate: 51.308%
Test Blue win rate:  51.897%

Option B — chronological 80/20
Train Blue win rate: 51.399%
Test Blue win rate:  51.721%


In [14]:

# PREPROCESSING STEP 1 — create and validate the approved patch-aligned split

import pandas as pd

PATCH_TEST_START = "7.10.187.9675"
champion_cols = [f"championid_p{i}" for i in range(1, 11)]

# First timestamp belonging to the selected test patch
patch_cutoff = match_level.loc[
    match_level["version"] == PATCH_TEST_START, "creation"
].min()

if pd.isna(patch_cutoff):
    raise ValueError(f"Test patch {PATCH_TEST_START} was not found.")

# Chronological split: all matches before the patch are train;
# the selected patch and later matches are test.
train_matches_patch = match_level[
    match_level["creation"] < patch_cutoff
].copy()

test_matches_patch = match_level[
    match_level["creation"] >= patch_cutoff
].copy()

# Validate expected sizes and disjointness
train_ids = set(train_matches_patch["matchid"])
test_ids = set(test_matches_patch["matchid"])

print("1. SPLIT SIZES")
print(f"Training matches: {len(train_matches_patch):,}")
print(f"Test matches:     {len(test_matches_patch):,}")
print(f"Total:            {len(train_matches_patch) + len(test_matches_patch):,}")
print(f"Expected train:   118,675")
print(f"Expected test:     42,399")

print("\n2. SPLIT INTEGRITY")
print("Match IDs overlap:", len(train_ids & test_ids))
print("Matches missing from both sets:",
      len(set(match_level["matchid"]) - train_ids - test_ids))
print("Training dates:",
      pd.to_datetime(train_matches_patch["creation"].min(), unit="ms", utc=True),
      "to",
      pd.to_datetime(train_matches_patch["creation"].max(), unit="ms", utc=True))
print("Test dates:",
      pd.to_datetime(test_matches_patch["creation"].min(), unit="ms", utc=True),
      "to",
      pd.to_datetime(test_matches_patch["creation"].max(), unit="ms", utc=True))

print("\n3. PATCH MIX IN TEST")
display(
    test_matches_patch["version"]
    .value_counts()
    .rename_axis("version")
    .to_frame("matches")
    .sort_index()
)

print("\n4. TARGET DISTRIBUTION")
display(
    pd.DataFrame({
        "train_blue_win_rate_pct": [
            train_matches_patch["blue_win"].mean() * 100
        ],
        "test_blue_win_rate_pct": [
            test_matches_patch["blue_win"].mean() * 100
        ],
        "train_missing_targets": [
            train_matches_patch["blue_win"].isna().sum()
        ],
        "test_missing_targets": [
            test_matches_patch["blue_win"].isna().sum()
        ],
    }).round(2)
)

print("\n5. DUPLICATE CHAMPION PICK AUDIT")
def count_duplicate_pick_matches(df):
    values = df[champion_cols].to_numpy()
    return sum(len(set(row)) < len(row) for row in values)

print("Training matches with repeated champion IDs:",
      count_duplicate_pick_matches(train_matches_patch))
print("Test matches with repeated champion IDs:",
      count_duplicate_pick_matches(test_matches_patch))

# Fail loudly if the approved split differs from the inspected candidate.
assert len(train_matches_patch) == 118675, "Unexpected training size."
assert len(test_matches_patch) == 42399, "Unexpected test size."
assert not (train_ids & test_ids), "Train/test match ID overlap detected."
assert train_matches_patch["blue_win"].notna().all()
assert test_matches_patch["blue_win"].notna().all()

print("\nSplit validation passed.")


1. SPLIT SIZES
Training matches: 118,675
Test matches:     42,399
Total:            161,074
Expected train:   118,675
Expected test:     42,399

2. SPLIT INTEGRITY
Match IDs overlap: 0
Matches missing from both sets: 0
Training dates: 2016-11-21 18:47:46.334000+00:00 to 2017-05-17 04:04:26.205000+00:00
Test dates: 2017-05-17 04:45:11.581000+00:00 to 2017-06-08 08:02:21.036000+00:00

3. PATCH MIX IN TEST


,matches
version,
7.10.187.9675,42379
7.11.189.942,20



4. TARGET DISTRIBUTION


,train_blue_win_rate_pct,test_blue_win_rate_pct,train_missing_targets,test_missing_targets
0,51.31,51.9,0,0



5. DUPLICATE CHAMPION PICK AUDIT
Training matches with repeated champion IDs: 0
Test matches with repeated champion IDs: 0

Split validation passed.


In [15]:

# PREPROCESSING STEP 2 — build unordered team-composition tables
# No encoding, feature scaling, or model training in this step.

def build_team_compositions(matches_df, players_df):
    match_ids = set(matches_df["matchid"])

    players = players_df.loc[
        players_df["matchid"].isin(match_ids),
        ["matchid", "player", "championid", "ss1", "ss2", "win"]
    ].copy()

    players["team_side"] = players["player"].map(
        lambda p: "blue" if p <= 5 else "red"
    )

    # Each team's picks are sorted by champion ID, so composition order
    # does not depend on the original participant-slot order.
    rows = []

    for (match_id, side), group in players.groupby(
        ["matchid", "team_side"], sort=False
    ):
        ordered_group = group.sort_values("championid")

        rows.append({
            "matchid": match_id,
            "team_side": side,
            "champions": tuple(ordered_group["championid"].astype(int)),
            # Spell values remain associated with their champion.
            "champion_spell_records": tuple(
                (
                    int(row.championid),
                    int(row.ss1),
                    int(row.ss2)
                )
                for row in ordered_group.itertuples(index=False)
            ),
            "pick_count": len(ordered_group),
            "unique_champions": ordered_group["championid"].nunique(),
            "team_win": group["win"].mean()
        })

    return pd.DataFrame(rows)


train_team_compositions = build_team_compositions(
    train_matches_patch, clean_players
)
test_team_compositions = build_team_compositions(
    test_matches_patch, clean_players
)

print("1. COMPOSITION TABLE SIZES")
print("Train team rows:", len(train_team_compositions))
print("Test team rows:", len(test_team_compositions))
print("Expected train rows:", len(train_matches_patch) * 2)
print("Expected test rows:", len(test_matches_patch) * 2)

print("\n2. STRUCTURAL VALIDATION")
for name, compositions in [
    ("Train", train_team_compositions),
    ("Test", test_team_compositions)
]:
    print(f"\n{name}")
    print("Rows per match:",
          compositions.groupby("matchid").size().value_counts().to_dict())
    print("Team pick counts:",
          compositions["pick_count"].value_counts().to_dict())
    print("Unique champion counts:",
          compositions["unique_champions"].value_counts().to_dict())
    print("Missing champion/spell records:",
          int(compositions["champion_spell_records"].isna().sum()))

print("\n3. SAMPLE COMPOSITIONS")
display(
    train_team_compositions[
        ["matchid", "team_side", "champions",
         "champion_spell_records", "team_win"]
    ].head(4)
)

print("\n4. CHECK MATCH-LEVEL TARGET AGREEMENT")
target_lookup = match_level.set_index("matchid")["blue_win"]

def target_agreement(compositions):
    blue = compositions[compositions["team_side"] == "blue"].set_index("matchid")
    red = compositions[compositions["team_side"] == "red"].set_index("matchid")

    shared_ids = blue.index.intersection(red.index)
    blue_matches_target = blue.loc[shared_ids, "team_win"].eq(
        target_lookup.loc[shared_ids]
    )
    red_matches_target = red.loc[shared_ids, "team_win"].eq(
        1 - target_lookup.loc[shared_ids]
    )

    return int((blue_matches_target & red_matches_target).sum()), len(shared_ids)

print("Train agreement (matches checked):",
      target_agreement(train_team_compositions))
print("Test agreement (matches checked):",
      target_agreement(test_team_compositions))

assert len(train_team_compositions) == len(train_matches_patch) * 2
assert len(test_team_compositions) == len(test_matches_patch) * 2
assert (train_team_compositions["pick_count"] == 5).all()
assert (test_team_compositions["pick_count"] == 5).all()
assert (train_team_compositions["unique_champions"] == 5).all()
assert (test_team_compositions["unique_champions"] == 5).all()

assert target_agreement(train_team_compositions)[0] == len(train_matches_patch)
assert target_agreement(test_team_compositions)[0] == len(test_matches_patch)

print("\nTeam-composition validation passed.")


1. COMPOSITION TABLE SIZES
Train team rows: 237350
Test team rows: 84798
Expected train rows: 237350
Expected test rows: 84798

2. STRUCTURAL VALIDATION

Train
Rows per match: {2: 118675}
Team pick counts: {5: 237350}
Unique champion counts: {5: 237350}
Missing champion/spell records: 0

Test
Rows per match: {2: 42399}
Team pick counts: {5: 84798}
Unique champion counts: {5: 84798}
Missing champion/spell records: 0

3. SAMPLE COMPOSITIONS


,matchid,team_side,champions,champion_spell_records,team_win
0,19,blue,"(40, 84, 96, 113, 420)","((40, 4, 3), (84, 14, 4), (96, 7, 4), (113, 4,...",1.0
1,19,red,"(19, 53, 58, 127, 236)","((19, 4, 11), (53, 3, 4), (58, 4, 12), (127, 4...",0.0
2,20,blue,"(19, 40, 55, 62, 236)","((19, 4, 11), (40, 4, 3), (55, 4, 14), (62, 12...",0.0
3,20,red,"(11, 75, 245, 497, 498)","((11, 11, 6), (75, 4, 12), (245, 4, 14), (497,...",1.0



4. CHECK MATCH-LEVEL TARGET AGREEMENT
Train agreement (matches checked): (118675, 118675)
Test agreement (matches checked): (42399, 42399)

Team-composition validation passed.


In [16]:

import numpy as np
import pandas as pd

# PREPROCESSING STEP 3 — encode champion indicators and team-level spell counts
# No model training, scaling, or feature selection.

def build_match_features(compositions, match_metadata, champion_ids):
    champion_ids = sorted(int(champ) for champ in champion_ids)

    blue = (
        compositions[compositions["team_side"] == "blue"]
        .set_index("matchid")
    )
    red = (
        compositions[compositions["team_side"] == "red"]
        .set_index("matchid")
    )

    match_ids = blue.index.intersection(red.index)
    blue = blue.loc[match_ids]
    red = red.loc[match_ids]

    # Champion-only representation: one indicator per champion per side.
    features = pd.DataFrame(index=match_ids)

    for side, team in [("blue", blue), ("red", red)]:
        for champ_id in champion_ids:
            features[f"{side}_champ_{champ_id}"] = team["champions"].map(
                lambda picks: int(champ_id in picks)
            )

    # Add team-level counts for each summoner spell ID.
    spell_ids = sorted({
        spell
        for records in compositions["champion_spell_records"]
        for record in records
        for spell in record[1:]
    })

    for side, team in [("blue", blue), ("red", red)]:
        for spell_id in spell_ids:
            features[f"{side}_spell_{spell_id}"] = team[
                "champion_spell_records"
            ].map(
                lambda records: sum(
                    spell == spell_id
                    for record in records
                    for spell in record[1:]
                )
            )

    metadata = (
        match_metadata
        .set_index("matchid")
        .loc[match_ids, ["version", "creation", "blue_win"]]
        .copy()
    )

    # Keep target and audit metadata separate from the predictors.
    y = metadata["blue_win"].astype("int8").copy()
    audit = metadata[["version", "creation"]].copy()

    champion_feature_cols = [
        col for col in features.columns if "_champ_" in col
    ]
    spell_feature_cols = [
        col for col in features.columns if "_spell_" in col
    ]

    X_champion = features[champion_feature_cols].astype("int8")
    X_champion_spells = features[
        champion_feature_cols + spell_feature_cols
    ].astype("int8")

    return X_champion, X_champion_spells, y, audit


# Use the observed champion IDs from the validated champion lookup.
observed_champion_ids = sorted(
    clean_players["championid"].dropna().astype(int).unique()
)

X_train_champion, X_train_champion_spells, y_train, audit_train = (
    build_match_features(
        train_team_compositions,
        train_matches_patch,
        observed_champion_ids
    )
)

X_test_champion, X_test_champion_spells, y_test, audit_test = (
    build_match_features(
        test_team_compositions,
        test_matches_patch,
        observed_champion_ids
    )
)

print("FEATURE TABLE SHAPES")
print("Train champion-only:", X_train_champion.shape)
print("Train champion + spells:", X_train_champion_spells.shape)
print("Train target:", y_train.shape)
print("Test champion-only:", X_test_champion.shape)
print("Test champion + spells:", X_test_champion_spells.shape)
print("Test target:", y_test.shape)

print("\nTARGET DISTRIBUTION")
print("Train:", y_train.value_counts().sort_index().to_dict())
print("Test:", y_test.value_counts().sort_index().to_dict())

print("\nFEATURE VALIDATION")
print("Missing train features:", int(X_train_champion_spells.isna().sum().sum()))
print("Missing test features:", int(X_test_champion_spells.isna().sum().sum()))
print("Train/test champion-only columns match:",
      X_train_champion.columns.equals(X_test_champion.columns))
print("Train/test champion + spell columns match:",
      X_train_champion_spells.columns.equals(
          X_test_champion_spells.columns
      ))
print("Train/test match IDs overlap:",
      len(set(audit_train.index) & set(audit_test.index)))
print("Train target rate:", round(float(y_train.mean()), 4))
print("Test target rate:", round(float(y_test.mean()), 4))

assert X_train_champion.shape[1] == 2 * len(observed_champion_ids)
assert X_train_champion.columns.equals(X_test_champion.columns)
assert X_train_champion_spells.columns.equals(
    X_test_champion_spells.columns
)
assert X_train_champion_spells.isna().sum().sum() == 0
assert X_test_champion_spells.isna().sum().sum() == 0
assert y_train.index.equals(X_train_champion.index)
assert y_test.index.equals(X_test_champion.index)
assert audit_train.index.equals(X_train_champion.index)
assert audit_test.index.equals(X_test_champion.index)
assert len(set(audit_train.index) & set(audit_test.index)) == 0

print("\nFeature construction and validation passed.")


C:\Users\Administrator\AppData\Local\Temp\ipykernel_2172\2162346139.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"{side}_champ_{champ_id}"] = team["champions"].map(
C:\Users\Administrator\AppData\Local\Temp\ipykernel_2172\2162346139.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"{side}_champ_{champ_id}"] = team["champions"].map(
C:\Users\Administrator\AppData\Local\Temp\ipykernel_2172\2162346139.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `f

FEATURE TABLE SHAPES
Train champion-only: (118675, 272)
Train champion + spells: (118675, 290)
Train target: (118675,)
Test champion-only: (42399, 272)
Test champion + spells: (42399, 290)
Test target: (42399,)

TARGET DISTRIBUTION
Train: {0: 57785, 1: 60890}
Test: {0: 20395, 1: 22004}

FEATURE VALIDATION
Missing train features: 0
Missing test features: 0
Train/test champion-only columns match: True
Train/test champion + spell columns match: True
Train/test match IDs overlap: 0
Train target rate: 0.5131
Test target rate: 0.519

Feature construction and validation passed.


In [17]:

# PREPROCESSING STEP 4 — side-symmetry validation
# Validation only: no feature changes and no model training.

def check_side_symmetry(X, y, feature_label):
    swapped = X.copy()

    for col in X.columns:
        if col.startswith("blue_"):
            red_col = "red_" + col[len("blue_"):]
            if red_col in X.columns:
                swapped[col] = X[red_col].to_numpy()
        elif col.startswith("red_"):
            blue_col = "blue_" + col[len("red_"):]
            if blue_col in X.columns:
                swapped[col] = X[blue_col].to_numpy()

    # Confirm the swap actually exchanged every corresponding side feature.
    side_cols = [
        col for col in X.columns
        if col.startswith(("blue_", "red_"))
    ]
    changed_rows = int(
        (swapped[side_cols].to_numpy() != X[side_cols].to_numpy())
        .any(axis=1).sum()
    )

    # The original label describes whether Blue won.
    # After swapping sides, the equivalent label is its complement.
    expected_swapped_target = 1 - y

    print(f"\n{feature_label}")
    print("Feature columns:", X.shape[1])
    print("Rows whose features changed after swapping:", changed_rows)
    print("Original Blue-win rate:", round(float(y.mean()), 4))
    print(
        "Expected Blue-win rate after swapping:",
        round(float(expected_swapped_target.mean()), 4)
    )
    print("Target inversion valid:",
          bool(expected_swapped_target.equals(1 - y)))

    assert len(side_cols) == X.shape[1]
    assert changed_rows > 0
    assert expected_swapped_target.isin([0, 1]).all()

    return swapped, expected_swapped_target


X_train_champion_swapped, y_train_swapped = check_side_symmetry(
    X_train_champion, y_train, "TRAIN — champion-only"
)

X_test_champion_swapped, y_test_swapped = check_side_symmetry(
    X_test_champion, y_test, "TEST — champion-only"
)

X_train_spells_swapped, _ = check_side_symmetry(
    X_train_champion_spells, y_train, "TRAIN — champion + spells"
)

X_test_spells_swapped, _ = check_side_symmetry(
    X_test_champion_spells, y_test, "TEST — champion + spells"
)

print("\nSide-symmetry structural checks passed.")



TRAIN — champion-only
Feature columns: 272
Rows whose features changed after swapping: 118675
Original Blue-win rate: 0.5131
Expected Blue-win rate after swapping: 0.4869
Target inversion valid: True

TEST — champion-only
Feature columns: 272
Rows whose features changed after swapping: 42399
Original Blue-win rate: 0.519
Expected Blue-win rate after swapping: 0.481
Target inversion valid: True

TRAIN — champion + spells
Feature columns: 290
Rows whose features changed after swapping: 118675
Original Blue-win rate: 0.5131
Expected Blue-win rate after swapping: 0.4869
Target inversion valid: True

TEST — champion + spells
Feature columns: 290
Rows whose features changed after swapping: 42399
Original Blue-win rate: 0.519
Expected Blue-win rate after swapping: 0.481
Target inversion valid: True

Side-symmetry structural checks passed.


In [18]:

# PREPROCESSING STEP 5 — final feature audit
# Read-only validation. No model training.

def audit_feature_tables(X_champ, X_spells, y, audit, split_name):
    champion_cols = [c for c in X_champ.columns if "_champ_" in c]
    spell_cols = [c for c in X_spells.columns if "_spell_" in c]

    print(f"\n{split_name}")

    # Champion indicators must be binary.
    champion_values = set(np.unique(X_champ.to_numpy()))
    print("Champion values:", sorted(champion_values))
    print("Champion indicators binary:", champion_values.issubset({0, 1}))

    # Every match should contain five champions on each side.
    blue_champ_cols = [c for c in champion_cols if c.startswith("blue_")]
    red_champ_cols = [c for c in champion_cols if c.startswith("red_")]
    blue_picks = X_champ[blue_champ_cols].sum(axis=1)
    red_picks = X_champ[red_champ_cols].sum(axis=1)

    print("Blue picks per match:", blue_picks.value_counts().to_dict())
    print("Red picks per match:", red_picks.value_counts().to_dict())

    # Spell features count each spell among a team's ten spell selections.
    spell_values = X_spells[spell_cols].to_numpy()
    spell_valid = (
        np.issubdtype(spell_values.dtype, np.integer)
        and spell_values.min() >= 0
        and spell_values.max() <= 5
    )
    print("Spell values are integer counts in [0, 5]:", spell_valid)

    blue_spell_cols = [c for c in spell_cols if c.startswith("blue_")]
    red_spell_cols = [c for c in spell_cols if c.startswith("red_")]
    blue_spell_total = X_spells[blue_spell_cols].sum(axis=1)
    red_spell_total = X_spells[red_spell_cols].sum(axis=1)

    print("Blue spell selections per match:",
          blue_spell_total.value_counts().to_dict())
    print("Red spell selections per match:",
          red_spell_total.value_counts().to_dict())

    print("Target values:", sorted(y.unique().tolist()))
    print("Metadata columns:", audit.columns.tolist())
    print("Target and metadata excluded from features:",
          "blue_win" not in X_spells.columns
          and "version" not in X_spells.columns
          and "creation" not in X_spells.columns)
    print("Index alignment:", X_spells.index.equals(y.index)
          and X_spells.index.equals(audit.index))

    assert champion_values.issubset({0, 1})
    assert (blue_picks == 5).all() and (red_picks == 5).all()
    assert spell_valid
    assert (blue_spell_total == 10).all()
    assert (red_spell_total == 10).all()
    assert set(y.unique()).issubset({0, 1})
    assert "blue_win" not in X_spells.columns
    assert "version" not in X_spells.columns
    assert "creation" not in X_spells.columns
    assert X_spells.index.equals(y.index)
    assert X_spells.index.equals(audit.index)


audit_feature_tables(
    X_train_champion, X_train_champion_spells,
    y_train, audit_train, "TRAIN"
)

audit_feature_tables(
    X_test_champion, X_test_champion_spells,
    y_test, audit_test, "TEST"
)

print("\nFinal preprocessing audit passed.")



TRAIN
Champion values: [np.int8(0), np.int8(1)]
Champion indicators binary: True
Blue picks per match: {5: 118675}
Red picks per match: {5: 118675}
Spell values are integer counts in [0, 5]: True
Blue spell selections per match: {10: 118675}
Red spell selections per match: {10: 118675}
Target values: [0, 1]
Metadata columns: ['version', 'creation']
Target and metadata excluded from features: True
Index alignment: True

TEST
Champion values: [np.int8(0), np.int8(1)]
Champion indicators binary: True
Blue picks per match: {5: 42399}
Red picks per match: {5: 42399}
Spell values are integer counts in [0, 5]: True
Blue spell selections per match: {10: 42399}
Red spell selections per match: {10: 42399}
Target values: [0, 1]
Metadata columns: ['version', 'creation']
Target and metadata excluded from features: True
Index alignment: True

Final preprocessing audit passed.


In [19]:

# PRE-MODELING INSPECTION — read-only
# No preprocessing changes and no model training.

from importlib.metadata import version, PackageNotFoundError

expected_objects = [
    "X_train_champion",
    "X_train_champion_spells",
    "y_train",
    "audit_train",
    "X_test_champion",
    "X_test_champion_spells",
    "y_test",
    "audit_test",
    "train_matches_patch",
    "test_matches_patch",
    "train_team_compositions",
    "test_team_compositions",
    "match_level",
    "patch_cutoff",
]

print("1. EXPECTED NOTEBOOK OBJECTS")
for name in expected_objects:
    if name not in globals():
        print(f"{name}: MISSING")
        continue

    obj = globals()[name]
    shape = getattr(obj, "shape", None)
    print(f"{name}: available; type={type(obj).__name__}; shape={shape}")

print("\n2. ML LIBRARY VERSIONS")
for package in ["scikit-learn", "pandas", "numpy", "scipy"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\n3. SPLIT AND FEATURE ALIGNMENT")
required = [
    "X_train_champion", "X_train_champion_spells", "y_train",
    "audit_train", "X_test_champion", "X_test_champion_spells",
    "y_test", "audit_test"
]

if all(name in globals() for name in required):
    print("Train indices aligned:",
          X_train_champion.index.equals(y_train.index)
          and X_train_champion.index.equals(audit_train.index)
          and X_train_champion.index.equals(
              X_train_champion_spells.index
          ))

    print("Test indices aligned:",
          X_test_champion.index.equals(y_test.index)
          and X_test_champion.index.equals(audit_test.index)
          and X_test_champion.index.equals(
              X_test_champion_spells.index
          ))

    print("Train/test match IDs disjoint:",
          set(audit_train.index).isdisjoint(set(audit_test.index)))

    print("Feature columns identical across splits:",
          X_train_champion.columns.equals(X_test_champion.columns)
          and X_train_champion_spells.columns.equals(
              X_test_champion_spells.columns
          ))
else:
    print("Alignment checks skipped: one or more objects are missing.")

print("\n4. VALIDATION BOUNDARY")
if "patch_cutoff" in globals():
    print("Patch cutoff:", patch_cutoff)
else:
    print("patch_cutoff is unavailable.")

print("\nInspection complete. No models were trained.")


1. EXPECTED NOTEBOOK OBJECTS
X_train_champion: available; type=DataFrame; shape=(118675, 272)
X_train_champion_spells: available; type=DataFrame; shape=(118675, 290)
y_train: available; type=Series; shape=(118675,)
audit_train: available; type=DataFrame; shape=(118675, 2)
X_test_champion: available; type=DataFrame; shape=(42399, 272)
X_test_champion_spells: available; type=DataFrame; shape=(42399, 290)
y_test: available; type=Series; shape=(42399,)
audit_test: available; type=DataFrame; shape=(42399, 2)
train_matches_patch: available; type=DataFrame; shape=(118675, 40)
test_matches_patch: available; type=DataFrame; shape=(42399, 40)
train_team_compositions: available; type=DataFrame; shape=(237350, 7)
test_team_compositions: available; type=DataFrame; shape=(84798, 7)
match_level: available; type=DataFrame; shape=(161074, 40)
patch_cutoff: available; type=int64; shape=()

2. ML LIBRARY VERSIONS
scikit-learn: 1.9.1
pandas: 2.3.3
numpy: 2.2.6
scipy: 1.13.1

3. SPLIT AND FEATURE ALIGNMENT

In [20]:

# Save validated preprocessing artifacts for Notebook 03.
# No model training.

from pathlib import Path
import joblib

artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

artifacts = {
    "X_train_champion": X_train_champion,
    "X_train_champion_spells": X_train_champion_spells,
    "y_train": y_train,
    "audit_train": audit_train,
    "X_test_champion": X_test_champion,
    "X_test_champion_spells": X_test_champion_spells,
    "y_test": y_test,
    "audit_test": audit_test,
}

artifact_path = artifact_dir / "nexusiq_preprocessed.joblib"
joblib.dump(artifacts, artifact_path, compress=3)

print("Saved:", artifact_path.resolve())
print("Artifact size (MB):", round(artifact_path.stat().st_size / (1024**2), 2))
print("Saved objects:", list(artifacts.keys()))


Saved: C:\Users\Administrator\Self Study\NexusIQ\notebooks\artifacts\nexusiq_preprocessed.joblib
Artifact size (MB): 7.58
Saved objects: ['X_train_champion', 'X_train_champion_spells', 'y_train', 'audit_train', 'X_test_champion', 'X_test_champion_spells', 'y_test', 'audit_test']
